<a href="https://colab.research.google.com/github/oit-cpai/cpai-foundations/blob/main/solutions/02_probability_expectation_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# M2 確率・期待値・標本平均(notebook)

本文:[`02-probability-expectation.md`](https://github.com/oit-cpai/cpai-foundations/blob/main/math/02-probability-expectation.md)

**進め方**:予想 → 手計算 → コードで確かめる → 実装(`TODO`)→ 条件を変える → 説明。`check(...)` が `OK` と出るまで直してから次に進む。上のセルから順に実行すること。

> **これは解答つきの版である。** まず `math/` にある演習の版を自分で解き、答え合わせに使うこと。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def check(name, got, expected, tol=1e-9):
    got, expected = np.asarray(got, dtype=float), np.asarray(expected, dtype=float)
    if got.shape != expected.shape:
        print(f"NG  {name}: 形が違う  got {got.shape}, expected {expected.shape}")
    elif np.allclose(got, expected, atol=tol):
        print(f"OK  {name}")
    else:
        print(f"NG  {name}: got {got}, expected {expected}")

## 1〜2. 分布から期待値と分散を計算する

さいころを振って、1・2・3 の目なら 0 点、4・5 なら 3 点、6 なら 6 点もらえる。点数 $X$ のとりうる値は $\{0, 3, 6\}$。

**手計算**:$p(0), p(3), p(6)$ と、$\mathbb{E}[X]$、$\mathrm{Var}[X]$ を紙で求める。

- 手計算:(ここに書く)

**実装**:値の並び `values` と確率の並び `probs` から、期待値と分散を計算する関数を書く(`for` 文でも、M1 の内積 `@` でもよい)。

In [ ]:
values = np.array([0.0, 3.0, 6.0])
probs = np.array([3/6, 2/6, 1/6])
check("確率の和は 1", probs.sum(), 1.0)

def expectation(values, probs):
    return values @ probs

def variance(values, probs):
    m = expectation(values, probs)
    return ((values - m) ** 2) @ probs

check("期待値", expectation(values, probs), 2.0)
check("分散", variance(values, probs), 5.0)

## 3. 標本平均は、試すたびに変わる

この分布から $N$ 回サンプルを取り、標本平均 $\bar{x}_N$ を計算する。

**予想**:$N = 10$ の標本平均は、だいたいどのくらいの範囲に入るか。$N = 1000$ ではどうか。

- 予想:(ここに書く)

**実装**:`rng.choice` で分布からサンプルを取り、標本平均を返す関数を書く。

In [ ]:
def sample_mean(values, probs, N, rng):
    x = rng.choice(values, size=N, p=probs)
    return x.mean()

rng = np.random.default_rng(0)
for N in [10, 100, 1000, 10000]:
    print(f"N={N:6d}: 標本平均 = {sample_mean(values, probs, N, rng):.3f}   (期待値 2.0)")

同じ $N$ でも、試すたびに標本平均は変わる。次のセルでは、5本の系列(seed 違い)について、$N$ を増やしながら標本平均を描く。

In [ ]:
Ns = np.arange(1, 2001)
plt.figure(figsize=(7, 3.5))
for seed in range(5):
    r = np.random.default_rng(seed)
    x = r.choice(values, size=Ns[-1], p=probs)
    plt.plot(Ns, np.cumsum(x) / Ns, lw=1)
plt.axhline(expectation(values, probs), color="k", ls="--", label="E[X]")
plt.xscale("log"); plt.xlabel("N (number of samples)"); plt.ylabel("sample mean"); plt.legend(); plt.show()

**ばらつきは $\sigma / \sqrt{N}$**:標本平均を何度も(1000回)計算し、その標準偏差を $\sigma/\sqrt{N}$ と比べる。

In [ ]:
sigma = np.sqrt(variance(values, probs))
rng = np.random.default_rng(1)
for N in [10, 40, 160, 640]:
    means = [sample_mean(values, probs, N, rng) for _ in range(1000)]
    print(f"N={N:4d}: 標本平均の標準偏差 {np.std(means):.3f}   理論 σ/√N = {sigma / np.sqrt(N):.3f}")

$N$ を4倍にするごとに、ばらつきが半分になっていることを確かめる。

## 4. 条件付き確率を表から計算する

赤玉6個(印つき2個)、白玉4個(印つき3個)。同時確率の表 `joint[色, 印]` を作り、$p(\text{赤} \mid \text{印つき})$ を計算する。

**手計算**:(ここに書く)

**実装**:同時確率の表から、条件付き確率 $p(\text{色} \mid \text{印})$ の表を計算する関数を書く。

In [ ]:
# 行:色(0=赤, 1=白)、列:印(0=なし, 1=あり)。全体 10 個で割って確率にする
counts = np.array([[4, 2],
                   [1, 3]])
joint = counts / counts.sum()
check("同時確率の和は 1", joint.sum(), 1.0)

def conditional_color_given_mark(joint):
    # 返り値 cond[色, 印] = p(色 | 印)。各列の和が 1 になる
    p_mark = joint.sum(axis=0)          # 印ごとの確率(色について足す)
    return joint / p_mark

cond = conditional_color_given_mark(joint)
check("p(赤 | 印あり) = 2/5", cond[0, 1], 2 / 5)
check("各列の和が 1", cond.sum(axis=0), [1.0, 1.0])

## 5. 数えて遷移確率を推定する

M1 で使った3状態・2行動の環境で、ランダムに行動して経験を集め、

$$
\hat{P}(s' \mid s, a) = \frac{N(s, a, s')}{N(s, a)}
$$

で遷移確率を推定する。

**予想**:経験の数を増やすと、推定値と本当の値の差はどう変わるか。

- 予想:(ここに書く)

**実装**:経験の列 `(s, a, s2)` から、回数を数えて $\hat P$ を返す関数を書く。一度も試していない `(s, a)` は、一様な確率 $1/|S|$ にする。

In [ ]:
P_true = np.array([
    [[0.8, 0.2, 0.0], [0.1, 0.9, 0.0]],
    [[0.0, 0.5, 0.5], [0.3, 0.0, 0.7]],
    [[0.0, 0.0, 1.0], [0.0, 0.0, 1.0]],
])
S, A = 3, 2

def collect(n_steps, rng):
    # ランダムな行動で n_steps 回動き、(s, a, s2) の列を返す。状態2に着いたら状態0から始め直す
    data, s = [], 0
    for _ in range(n_steps):
        a = int(rng.integers(A))
        s2 = int(rng.choice(S, p=P_true[s, a]))
        data.append((s, a, s2))
        s = 0 if s2 == 2 else s2
    return data

def estimate_P(data, S, A):
    N_sas = np.zeros((S, A, S))
    for s, a, s2 in data:
        N_sas[s, a, s2] += 1
    N_sa = N_sas.sum(axis=2, keepdims=True)
    P_hat = np.where(N_sa > 0, N_sas / np.maximum(N_sa, 1), 1.0 / S)
    return P_hat

rng = np.random.default_rng(0)
P_hat = estimate_P(collect(5000, rng), S, A)
check("推定した確率の和は 1", P_hat.sum(axis=2), np.ones((S, A)))
print("状態0・行動0 の推定:", np.round(P_hat[0, 0], 3), " 本当:", P_true[0, 0])
print("状態2(ゴール)からは動かないので、一度も試していない →", P_hat[2, 0])

経験の数と、推定の誤差(本当の値との差の最大値)の関係を見る。

In [ ]:
rng = np.random.default_rng(1)
for n in [50, 200, 800, 3200, 12800]:
    P_hat = estimate_P(collect(n, rng), S, A)
    err = np.abs(P_hat[:2] - P_true[:2]).max()      # 試せる状態0・1 について
    print(f"経験 {n:6d} 回: 最大誤差 {err:.3f}")

経験を増やすと、誤差は小さくなる。1回ごとの結果はばらつくが、平均的には経験を4倍にすると誤差はおよそ半分になる。3. の $\sigma / \sqrt{N}$ と同じ理由である(確率の推定も、0 か 1 の値の標本平均だから)。seed を変えて何度か実行し、確かめる。

## 6. マルコフ連鎖:状態の分布の変化

2状態のマルコフ連鎖 $P = \begin{pmatrix} 0.9 & 0.1 \\ 0.5 & 0.5 \end{pmatrix}$ を考える。時刻 $t$ の分布 $d_t$(横ベクトル)は $d_{t+1} = d_t P$ で変わる。

**手計算**:$d_0 = (1, 0)$ のとき、$d_1$ と $d_2$ を求める。

- 手計算:(ここに書く)

**実装**:$d_0$ から $T$ ステップ分の分布を順に計算する関数を書く。

In [ ]:
P2 = np.array([[0.9, 0.1],
               [0.5, 0.5]])

def distributions(d0, P, T):
    ds = [np.asarray(d0, dtype=float)]
    for _ in range(T):
        ds.append(ds[-1] @ P)
    return np.array(ds)

ds = distributions([1.0, 0.0], P2, 30)
check("d_1", ds[1], [0.9, 0.1])
check("d_2", ds[2], [0.86, 0.14])

ds_b = distributions([0.0, 1.0], P2, 30)
plt.figure(figsize=(6, 3))
plt.plot(ds[:, 1], marker=".", label="start from state 0")
plt.plot(ds_b[:, 1], marker=".", label="start from state 1")
plt.xlabel("time t"); plt.ylabel("Pr(state 1)"); plt.legend(); plt.show()
print("30 ステップ後:", np.round(ds[-1], 4), np.round(ds_b[-1], 4))

どちらから始めても、同じ分布に落ち着く(**定常分布**。この例では $(5/6, 1/6)$)。M6 で詳しく扱う。

**マルコフ性と独立の違い**を見る:1本の系列をシミュレーションし、「今が状態1のとき、次も状態1である割合」と「全体で状態1である割合」を比べる。独立なら同じになるはずである。

In [ ]:
rng = np.random.default_rng(0)
s, seq = 0, []
for _ in range(20000):
    seq.append(s)
    s = int(rng.choice(2, p=P2[s]))
seq = np.array(seq)
now1 = seq[:-1] == 1
print("全体で状態1 の割合          :", round(seq.mean(), 3))
print("今が状態1 のとき、次も状態1 :", round(seq[1:][now1].mean(), 3), " ← P2[1,1] = 0.5 に近い")
print("今が状態0 のとき、次が状態1 :", round(seq[1:][~now1].mean(), 3), " ← P2[0,1] = 0.1 に近い")

## 7. 条件を変える

1. 1〜2 の点数を変えて(例:6 の目を 12 点に)、期待値・分散と、標本平均のばらつきがどう変わるかを見る
2. 5 で、行動をランダムではなく「いつも行動0」にすると、推定できない `(s, a)` が増えることを確かめる。強化学習の「探索」が必要な理由の1つである
3. 6 で、`P2` を $\begin{pmatrix} 0.5 & 0.5 \\ 0.5 & 0.5 \end{pmatrix}$ にすると、状態の列は独立になる。上のセルの3つの割合がどうなるかを予想してから確かめる

## 8. 説明(自分の言葉で書く)

1. 期待値と標本平均の違いを、さいころの例で説明する
   - (ここに書く)
2. 2つの方法を、seed 1つずつの結果で比べてはいけないのはなぜか
   - (ここに書く)
3. 遷移確率を数えて推定するとき、経験が少ないと何が困るか
   - (ここに書く)